# Smart Medic — RUNBOOK

**Đường thẳng từ repo sạch tới `output.zip`.** Chỉ các lệnh thực thi chính, đúng thứ tự.
Không giải thích thiết kế, không phân tích — những thứ đó ở
[`docs/reports/plan-v4.html`](../docs/reports/plan-v4.html).

Chạy **từ trên xuống**. Mỗi ô in `✓` hoặc `✗`; ô nào `✗` thì **dừng ở đó** — các ô sau không
có nghĩa nữa.

| Ô | Bước | Phase | Bắt buộc |
|---|---|---|---|
| 1 | Môi trường | — | ✓ |
| 2 | **Cổng toàn vẹn** — offset + `data/test/` chưa bị sửa | P0 | ✓ **không được bỏ** |
| 3 | Dựng chỉ mục KB | P1 · P5 | ✓ |
| 4 | Chạy trên **gold** → chấm điểm (ĐO) | P0–P6 | ✓ |
| 5 | Chạy trên **`data/test/`** → `data/output/` (NỘP) | P0–P6 | ✓ |
| 6 | Đóng gói `output.zip` + manifest | P0 · P7 | ✓ |
| 7 | Diễn tập tái lập (container sạch) | P7 | ✓ **chống bị loại** |
| 8 | **Tự kiểm** — notebook còn khớp repo không | — | ✓ sau **mỗi** phase |

> 🔁 **File này phải được chạy lại và cập nhật sau MỖI phase.** Mỗi phase thêm file mới và
> đôi khi đổi cờ dòng lệnh; ô 8 tự phát hiện chỗ lệch. Notebook trôi khỏi repo còn tệ hơn
> không có notebook — nó khiến người ta tin vào một lệnh đã sai.

> ⚠ Pipeline suy luận **chưa được cài** (P0–P6 còn ⬜). Ô 4 sẽ báo *chưa có* chứ không crash;
> mọi ô khác chạy được ngay hôm nay.

## 1 · Môi trường

In [1]:
import os, subprocess, sys, shutil, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

# PYTHONPATH=src là BẮT BUỘC: subprocess không thừa hưởng sys.path của notebook,
# nên `python3 -m smart_medic.…` sẽ không tìm thấy package nếu thiếu dòng này.
ENV = {**os.environ, "PYTHONPATH": str(ROOT / "src")}

def sh(cmd, *, must=True, mark=True):
    """Chạy lệnh ở gốc repo. must=True ⇒ ô dừng nếu lệnh fail."""
    p = subprocess.run(cmd, shell=True, cwd=ROOT, text=True, env=ENV,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout.rstrip()[:4000])
    if mark:
        print(("✓ " if p.returncode == 0 else "✗ ") + cmd)
    if must and p.returncode != 0:
        raise SystemExit(f"DỪNG — lệnh thất bại: {cmd}")
    return p.returncode, p.stdout

print("repo :", ROOT)
print("python:", sys.version.split()[0])
sh("git rev-parse --short HEAD && git status --porcelain | wc -l | xargs echo 'file bẩn:'")

repo : /Users/handonn/Workplace/AI/smart-medic
python: 3.13.9
8c00bd5
file bẩn: 35
✓ git rev-parse --short HEAD && git status --porcelain | wc -l | xargs echo 'file bẩn:'


(0, '8c00bd5\nfile bẩn: 35\n')

## 2 · CỔNG TOÀN VẸN — không được bỏ

Điểm tính trên offset đã lệch còn **tệ hơn không có điểm**: sai offset cho 0 điểm và
**hoàn toàn im lặng** (20/100 file test không ở NFC; chuẩn hoá trước khi tính offset làm lệch
tới 143 ký tự).

`test_silver_offsets` **FAIL là bình thường** — 165 vi phạm có thật trong corpus bạc, đã biết,
đã lọc lúc nạp. Mọi test khác phải xanh.

In [2]:
rc, out = sh("python3 -m pytest tests/ -q", must=False, mark=False)
fails = [l for l in out.splitlines() if l.startswith("FAILED")]
unexpected = [l for l in fails if "test_silver_offsets" not in l]
if unexpected:
    raise SystemExit("DỪNG — fail ngoài dự kiến:\n" + "\n".join(unexpected))
print("\n✓ cổng toàn vẹn: chỉ còn test_silver_offsets (đã biết)")

...................................................F.................... [ 83%]
..............                                                           [100%]
=================================== FAILURES ===================================
_____________________________ test_silver_offsets ______________________________

    def test_silver_offsets():
        """Generated training data must satisfy the same offset invariant."""
        pairs = list(_silver_pairs())
        if not pairs:
            pytest.skip("no silver corpus present")
    
        all_errs: list[str] = []
        bad_files = 0
        for kind, ann, txt in pairs:
            try:
                entities = json.loads(ann.read_text(encoding="utf-8"))
            except json.JSONDecodeError as exc:
                all_errs.append(f"{kind}/{ann.name}: invalid JSON — {exc}")
                bad_files += 1
                continue
            errs = check_entities(entities, read_raw(txt), f"{kind}/{ann.name}")
          

## 3 · Dựng chỉ mục KB

`data/knowledge_base/` là thư mục **phẳng**; đường dẫn và cách parse RRF nằm ở
`scripts/kb_sources.py`. Chỉ mục là cache — xoá đi thì lệnh này dựng lại.

In [3]:
sh("python3 -c \"import sys; sys.path.insert(0,'scripts'); import kb_sources as k;"
   " k.require(k.ICD10_VI, k.ICD10CM_EN, k.RXNCONSO, k.RXNREL, k.RXNSTY, k.RXNATOMARCHIVE);"
   " print('✓ 6 file KB có mặt')\"")
sh("python3 scripts/annotation_qa/kb.py build")

✓ 6 file KB có mặt
✓ python3 -c "import sys; sys.path.insert(0,'scripts'); import kb_sources as k; k.require(k.ICD10_VI, k.ICD10CM_EN, k.RXNCONSO, k.RXNREL, k.RXNSTY, k.RXNATOMARCHIVE); print('✓ 6 file KB có mặt')"


built: icd 13189 codes / 14240 names; icd_en 74879; rx 70245 cuis / 118118 names; b2i 96495
✓ python3 scripts/annotation_qa/kb.py build


(0,
 'built: icd 13189 codes / 14240 names; icd_en 74879; rx 70245 cuis / 118118 names; b2i 96495\n')

## 4 · Chạy trên **gold** → chấm điểm  ·  ĐO

Hai tập chạy khác nhau, đừng lẫn: **gold** có nhãn nên chấm được (đây là ô đo);
**`data/test/`** không có nhãn, nó là bài nộp (ô 5).

Số chính thức nội bộ là `penalised / greedy_iou`, nhưng scorer in **cả ba** cách đọc × **bốn**
chế độ căn chỉnh trong một lần chạy — phải đọc hết:

- **`greedy_iou`** — số chính thức. ⚠ Nó **không so trường `type`** ở đâu cả.
- **`overlap_type`** — **cột chặn**. Sai type 10% mất **12,35** điểm ở đây trong khi
  `greedy_iou` báo 0,00. Thay đổi chỉ được nhận nếu không làm cột này giảm quá **0,010**.
- **`exact`** — **đèn báo bug offset**. Sụt về gần 0 mà hai cột kia bình thường ⇒ bug offset,
  không phải khoảng trống mô hình.
- **`matched`** suy biến — xoá 30% dự đoán của chính mình làm nó *tăng*. Chỉ dùng làm trần.

In [4]:
GOLD_TXT = "data/generated_medical_records/restyled/text"
GOLD     = "data/generated_medical_records/restyled/annotations_gold"
PRED_GOLD = "runs/_pred_gold"

if not (ROOT / "src/smart_medic/cli.py").exists():
    print("⏳ CHƯA CÓ pipeline (P0–P6 còn ⬜). Chuỗi lệnh sẽ là:")
    print(f"   python3 -m smart_medic.cli run --input {GOLD_TXT} --output {PRED_GOLD}")
    print(f"   python3 -m smart_medic.eval.scoring --pred {PRED_GOLD} --gold {GOLD}")
else:
    sh(f"python3 -m smart_medic.cli run --input {GOLD_TXT} --output {PRED_GOLD}")
    sh(f"python3 -m smart_medic.eval.scoring --pred {PRED_GOLD} --gold {GOLD}")

⏳ CHƯA CÓ pipeline (P0–P6 còn ⬜). Chuỗi lệnh sẽ là:
   python3 -m smart_medic.cli run --input data/generated_medical_records/restyled/text --output runs/_pred_gold
   python3 -m smart_medic.eval.scoring --pred runs/_pred_gold --gold data/generated_medical_records/restyled/annotations_gold


## 5 · Chạy trên `data/test/` → `data/output/`  ·  NỘP

`data/test/` **bất biến** — hook chặn ghi, manifest sha256 là lớp hai.
`position` **luôn** index vào chuỗi gốc chưa chuẩn hoá.

In [5]:
CLI = "python3 -m smart_medic.cli run --input data/test --output data/output"
if not (ROOT / "src/smart_medic/cli.py").exists():
    print("⏳ CHƯA CÓ — P0–P6 còn ⬜. Lệnh sẽ là:\n   " + CLI)
    print("   Bắt đầu từ .claude/prompts/p0_prompt.md")
else:
    sh(CLI)
sh("python3 -m smart_medic.eval.scoring --pred data/output --describe", must=False)

⏳ CHƯA CÓ — P0–P6 còn ⬜. Lệnh sẽ là:
   python3 -m smart_medic.cli run --input data/test --output data/output
   Bắt đầu từ .claude/prompts/p0_prompt.md

── STRUCTURE of data/output ──
  documents            100
  entities             1585  (15.8 per doc)
  type distribution
      TRIỆU_CHỨNG               520   32.8%
      CHẨN_ĐOÁN                 420   26.5%
      THUỐC                     256   16.2%
      TÊN_XÉT_NGHIỆM            251   15.8%
      KẾT_QUẢ_XÉT_NGHIỆM        138    8.7%
  span length (words)  median 2  mean 2.19  max 10
  codes/entity (CĐ+thuốc) {0: 242, 1: 394, 2: 40}
  assertions/entity       {0: 1462, 1: 117, 2: 6}

(no --gold given, so no score. Hand-label a dev set and pass --gold.)
✓ python3 -m smart_medic.eval.scoring --pred data/output --describe


(0,
 '\n── STRUCTURE of data/output ──\n  documents            100\n  entities             1585  (15.8 per doc)\n  type distribution\n      TRIỆU_CHỨNG               520   32.8%\n      CHẨN_ĐOÁN                 420   26.5%\n      THUỐC                     256   16.2%\n      TÊN_XÉT_NGHIỆM            251   15.8%\n      KẾT_QUẢ_XÉT_NGHIỆM        138    8.7%\n  span length (words)  median 2  mean 2.19  max 10\n  codes/entity (CĐ+thuốc) {0: 242, 1: 394, 2: 40}\n  assertions/entity       {0: 1462, 1: 117, 2: 6}\n\n(no --gold given, so no score. Hand-label a dev set and pass --gold.)\n')

## 6 · Đóng gói `output.zip`

Đúng **100** file `1.json` … `100.json` trong thư mục `output/`, **0 file phụ**.
`unzip -l` phải được **dán nguyên** vào báo cáo — không mô tả bằng lời.

In [6]:
PKG = "python3 scripts/submit/package_submission.py"
if not (ROOT / "scripts/submit/package_submission.py").exists():
    print("⏳ CHƯA CÓ — P0 còn ⬜. Lệnh sẽ là:\n   " + PKG)
else:
    sh(PKG)
    sh("unzip -l output.zip")

run        : runs/2026-07-30T07-38_8c00bd5
predictions: data/output  ->  staging
  100 files · 1585 entities · density 15.85/file · 1585/1585 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes dropped 0 · dropped malformed 0 / bad type 0 / nested 0 / duplicate 0

✓ 7/7 archive checks passed
  output.zip  (53,663 bytes)
  runs/2026-07-30T07-38_8c00bd5/manifest.json

  !! WORKING TREE IS DIRTY — this build must NOT be submitted.
     runs/README.md: a dirty tree means the run cannot be reproduced.
✓ python3 scripts/submit/package_submission.py
Archive:  output.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
     7109  01-01-1980 00:00   output/1.json
     3449  01-01-1980 00:00   output/2.json
     5275  01-01-1980 00:00   output/3.json
    11095  01-01-1980 00:00   output/4.json
     2383  01-01-1980 00:00   output/5.json
     2976  01-01-1980 00:00   output/6.json
     2883  01-01-1980 00:00   output/7.json
     4212  01-01-1

## 7 · Diễn tập tái lập — **chống bị loại**

Rủi ro duy nhất **không mua lại được bằng điểm**: top ~15 đội nộp source + weights, ban tổ chức
chạy lại trên private test, **cài không được ⇒ bị loại**.

Chạy **hai lần**: lần đầu để tìm lỗi khi còn thời gian sửa, lần hai để xác nhận.
Diễn tập *sau* khi nộp là vô ích. Temperature 0 **không** cho tính xác định — nếu không
bit-identical thì cố định `batch_size=1`, còn lệch nữa thì đóng gói cache đầu ra kèm theo.

In [7]:
print("""Chạy NGOÀI notebook này, trên container sạch:

  docker run --rm -it -v "$PWD":/w -w /w python:3.13-slim bash -lc '
    pip install --require-hashes -r requirements.lock &&
    python3 -m smart_medic.cli run --input data/test --output /tmp/out &&
    python3 scripts/submit/package_submission.py --out /tmp/out.zip --run-dir /tmp/run --no-run-record'

  # rồi so với bản đã nộp. Bộ ghi archive là TẤT ĐỊNH (ADR 0005), nên cmp phải
  # SẠCH TUYỆT ĐỐI — một byte lệch là một dự đoán lệch, không phải nhiễu đóng gói.
  cmp /tmp/out.zip runs/<ts>_<sha>/output.zip && echo "✓ tái lập được"
""")
if (ROOT / "runs").exists():
    sh("ls -1 runs/ | tail -5", must=False)

Chạy NGOÀI notebook này, trên container sạch:

  docker run --rm -it -v "$PWD":/w -w /w python:3.13-slim bash -lc '
    pip install --require-hashes -r requirements.lock &&
    python3 -m smart_medic.cli run --input data/test --output /tmp/out &&
    python3 scripts/submit/package_submission.py --out /tmp/out.zip --run-dir /tmp/run --no-run-record'

  # rồi so với bản đã nộp. Bộ ghi archive là TẤT ĐỊNH (ADR 0005), nên cmp phải
  # SẠCH TUYỆT ĐỐI — một byte lệch là một dự đoán lệch, không phải nhiễu đóng gói.
  cmp /tmp/out.zip runs/<ts>_<sha>/output.zip && echo "✓ tái lập được"

2026-07-30T07-37_8c00bd5
2026-07-30T07-37_8c00bd5-2
2026-07-30T07-38_8c00bd5
README.md
leverage_map.json
✓ ls -1 runs/ | tail -5


## 8 · TỰ KIỂM — chạy sau **mỗi** phase

Notebook này nhân bản các lệnh của repo, nên nó **trôi được**. Ô dưới đây bắt chỗ trôi:
mọi đường dẫn nó nhắc tới phải tồn tại (hoặc là artifact P0–P7 đã biết là chưa có), và mọi
cờ dòng lệnh nó dùng phải thật sự có trong `--help`.

**Một phase chỉ được coi là xong khi ô này xanh.** Nếu đỏ: sửa notebook, đừng sửa cách đọc.

In [8]:
import re, json

NB = ROOT / "notebooks/runbook.ipynb"
cells = json.loads(NB.read_text())["cells"] if NB.exists() else []
src = "".join("".join(c["source"]) for c in cells if c["cell_type"] == "code")

PENDING = {  # artifact chưa có, thuộc phase nào — KHÔNG tính là lệch
    "src/smart_medic/cli.py": "P1–P6",
    "requirements.lock": "P7",
}
REQUIRED = [
    "scripts/kb_sources.py", "scripts/annotation_qa/kb.py",
    "scripts/analysis/measure_data.py", "scripts/analysis/leverage_map.py",
    "src/smart_medic/eval/scoring.py", "tests/test_offsets.py",
    "data/generated_medical_records/restyled/text",
    "data/generated_medical_records/restyled/annotations_gold",
    "data/test", ".claude/prompts/p0_prompt.md",
    # P0 · nền móng
    "configs/pipeline.yaml", "configs/models.yaml", "configs/metric.yaml",
    "src/smart_medic/io/document.py", "src/smart_medic/io/corpus.py",
    "src/smart_medic/layout/lines.py", "src/smart_medic/layout/outline.py",
    "src/smart_medic/layout/kv.py",
    "src/smart_medic/validate/schema.py", "src/smart_medic/validate/offsets.py",
    "src/smart_medic/validate/emit_json.py",
    "scripts/submit/package_submission.py",
    "tests/test_no_api_in_runtime.py", "tests/test_layer_boundaries.py",
    "pyproject.toml", "Makefile",
]

bad = [f"THIẾU (đáng lẽ phải có): {f}" for f in REQUIRED if not (ROOT / f).exists()]
for f, ph in PENDING.items():
    print(("✓ đã có    " if (ROOT / f).exists() else f"⏳ chờ {ph:7}") + f)

# Cờ dòng lệnh: chỉ soi các LỆNH THẬT gọi scorer, không soi chính ô kiểm tra này.
_, helptxt = sh("python3 -m smart_medic.eval.scoring --help", must=False, mark=False)
cmds = [m.group(0) for m in re.finditer(r'"[^"]*smart_medic\.eval\.scoring[^"]*"', src)]
used = {f for c in cmds for f in re.findall(r"--[a-z][a-z-]+", c)}
for flag in sorted(used):
    if flag not in helptxt:
        bad.append(f"NOTEBOOK DÙNG CỜ KHÔNG TỒN TẠI: {flag}  (trong: {[c for c in cmds if flag in c][0][:70]})")
# Cờ của BỘ ĐÓNG GÓI cũng phải thật. Ô 7 nhắc một lệnh chạy trong container, và
# một cờ sai ở đó chỉ lộ ra giữa buổi diễn tập tái lập — muộn nhất có thể.
PKG = ROOT / "scripts/submit/package_submission.py"
pkgused = set()
if PKG.exists():
    _, pkghelp = sh("python3 scripts/submit/package_submission.py --help",
                    must=False, mark=False)
    pkgcmds = re.findall(r"package_submission\.py[^\n\"']*", src)
    pkgused = {f for c in pkgcmds for f in re.findall(r"--[a-z][a-z-]+", c)}
    for flag in sorted(pkgused):
        if flag not in pkghelp:
            bad.append(f"NOTEBOOK DÙNG CỜ KHÔNG TỒN TẠI: {flag}  (package_submission.py)")

print()
if not bad:
    print("cờ scorer   :", sorted(used) or "—", " → đều hợp lệ")
    print("cờ đóng gói :", sorted(pkgused) or "—", " → đều hợp lệ")

if bad:
    print("✗ NOTEBOOK ĐÃ TRÔI KHỎI REPO:")
    for b in bad: print("   -", b)
    raise SystemExit("Sửa notebooks/runbook.ipynb trước khi coi phase là xong.")
print("✓ notebook còn khớp repo")

⏳ chờ P1–P6  src/smart_medic/cli.py
⏳ chờ P7     requirements.lock
usage: scoring.py [-h] [--pred PRED] [--gold GOLD] [--describe]
                  [--cand-formula {official,plain}] [--json]

Internal scorer for the Viettel AI Race 2026 Vòng-1 metric.

options:
  -h, --help            show this help message and exit
  --pred PRED
  --gold GOLD           directory of gold N.json files
  --describe            structural summary only, no gold needed
  --cand-formula {official,plain}
  --json                emit machine-readable JSON
usage: package_submission.py [-h] [--pred PRED] [--source SOURCE] [--out OUT]
                             [--expect EXPECT] [--probe PROBE] [--no-freeze]
                             [--no-kb-check] [--run-dir RUN_DIR]
                             [--no-run-record]

Build `output.zip` and the immutable run record. Build-time, not inference.

options:
  -h, --help         show this help message and exit
  --pred PRED
  --source SOURCE
  --out OUT
  --expect E

---

## Xong

`output.zip` + `runs/<ISO8601>_<git-sha7>/{output/, manifest.json, score.json}`.
Nộp bài **luôn là quyết định của con người** — 5 lần/ngày, không tự động hoá.

**Ba việc không được bỏ dù hết thời gian:** `pytest tests/test_offsets.py` sạch ·
ba test chống rò rỉ API xanh · `unzip -l output.zip` đúng 100 file trong thư mục `output/`.
Ba việc đó không cho thêm điểm nào, nhưng bỏ một trong ba là mất toàn bộ 70,00.

🔁 **Sau mỗi phase:** chạy lại notebook này từ đầu và cập nhật nếu ô 8 báo đỏ. Đây là một mục
trong tiêu chí nghiệm thu của mọi phase, không phải việc dọn dẹp tuỳ hứng.

<sub>Không nằm trên đường ra output, chạy khi cần: `python3 scripts/analysis/measure_data.py`
(số liệu corpus/KB) · `python3 scripts/analysis/leverage_map.py --seeds 6 --extras --json
runs/leverage_map.json` (bản đồ đòn bẩy).</sub>